# Titanic with Pairwise Difference Learning: raw DataFrame in, no feature engineering

**Pairwise Difference Learning (PDL)** trains a model on *pairs* of rows: "do these two rows have the same label?" (or, for regression, "how far apart are their targets?"). A new row is then compared with the training rows, whose labels are known, and the answers are combined. With n training rows the model sees up to n² examples, which helps most when data is scarce. [`pdll`](https://github.com/Karim-53/pdll) wraps any scikit-learn model with the usual `fit` / `predict` API. Method: [Belaid, Rabus & Hüllermeier, *Pairwise Difference Learning for Classification*, Discovery Science 2024](https://doi.org/10.1007/978-3-031-78980-9_18). *Disclosure: I am one of the authors.*

In [ ]:
!pip install -q -U pdll

In [ ]:
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

import pdll
from pdll import PairwiseDifferenceClassifier

DATA_DIR = os.environ.get('DATA_DIR', '/kaggle/input/titanic')
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
print('pdll', pdll.__version__, '| train', train.shape, '| test', test.shape)

Drop the identifiers and keep everything else as it is: strings (`Sex`, `Embarked`) and missing values (`Age`, `Fare`, `Embarked`). Category columns are understood by both the base model and pdll.

In [ ]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

def prepare(df):
    X = df[features].copy()
    for column in ('Sex', 'Embarked'):
        X[column] = X[column].astype('category')
    return X

X, y, X_test = prepare(train), train['Survived'], prepare(test)
X.head()

Same base model alone and wrapped in PDL, same 5-fold cross-validation. HistGradientBoosting handles the missing values natively.

In [ ]:
base = HistGradientBoostingClassifier(class_weight='balanced', categorical_features='from_dtype', random_state=0)
pdl = PairwiseDifferenceClassifier(base, random_state=0)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for name, model in [('HistGradientBoosting', base), ('PDL(HistGradientBoosting)', pdl)]:
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    print(f'{name:28} accuracy {scores.mean():.3f} ± {scores.std():.3f}')

Fit on all training passengers and write the submission.

In [ ]:
pdl.fit(X, y)
submission = pd.DataFrame({'PassengerId': test['PassengerId'], 'Survived': pdl.predict(X_test)})
submission.to_csv('submission.csv', index=False)
submission.head()

PDL also tells you how much the training passengers *disagree* about a prediction: `predict_proba_samples` gives the posterior from each training passenger (anchor) separately.

In [ ]:
per_anchor = pdl.predict_proba_samples(X_test[:5])[:, :, 1]   # P(survived) according to each anchor
pd.DataFrame({'P(survived)': pdl.predict_proba(X_test[:5])[:, 1].round(2),
              'spread across anchors': per_anchor.std(axis=1).round(2)}, index=test['PassengerId'][:5])

If PDL is useful in your work, please cite the paper:

```bibtex
@inproceedings{belaid2024pairwise,
  title     = {Pairwise Difference Learning for Classification},
  author    = {Belaid, Mohamed Karim and Rabus, Maximilian and H{\"u}llermeier, Eyke},
  booktitle = {Discovery Science},
  pages     = {284--299},
  publisher = {Springer},
  year      = {2024},
  doi       = {10.1007/978-3-031-78980-9_18}
}
```